# Benchmarking models for sulfur saturation

This notebook benchmarks the models for sulfur saturation in VolFe where possible.

## Python set-up

In [1]:
import pandas as pd
import VolFe as vf
import math

c:\Users\erych\miniconda3\envs\volfe_dev\Lib\site-packages\PySulfSat\__init__.py:8: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.2)
  import scipy.optimize as optimize


Version of VolFe

In [2]:
vf.__version__

'1.0.5'

## Composition and conditions used for benchmarking

Average of high-SiO2 pillow-rim glasses in HSDP from Mauna Kea volcano from Brounce et al. (2017).

In [3]:
my_analysis = {
        "Sample": "Hawaiian basalt",
        "T_C": 1200.0,  # Temperature in 'C
        "SiO2": 51.29,  # wt%
        "TiO2": 2.50,  # wt%
        "Al2O3": 13.70,  # wt%
        "FeOT": 11.04,  # wt%
        "MnO": 0.02,  # wt%
        "MgO": 6.70,  # wt%
        "CaO": 11.03,  # wt%
        "Na2O": 2.27,  # wt%
        "K2O": 0.43,  # wt%
        "P2O5": 0.21,  # wt%
        "H2O": 0.,  # wt%
        "CO2ppm": 0.,  # ppm
        "STppm": 0.,  # ppm
        "Xppm": 0.0,  # ppm
        "Fe3FeT": 0.1,
    }

my_analysis = pd.DataFrame(my_analysis, index=[0])

PT = {"P":1000.}
PT["T"]=1200.

melt_wf=vf.melt_comp(0.,my_analysis)
melt_wf['CO2'] = my_analysis.loc[0.,"CO2ppm"]/1000000.
melt_wf["H2OT"] = my_analysis.loc[0,"H2O"]/100.
melt_wf['ST'] = my_analysis.loc[0.,"STppm"]/1000000.
melt_wf['CT'] = (melt_wf['CO2']/vf.species.loc['CO2','M'])*vf.species.loc['C','M']
melt_wf['HT'] = (melt_wf['H2OT']/vf.species.loc['H2O','M'])*(2.*vf.species.loc['H','M'])
melt_wf['XT'] = my_analysis.loc[0.,"Xppm"]/1000000.
melt_wf["Fe3FeT"] = my_analysis.loc[0.,"Fe3FeT"]

## Models for Sulfide Content at Sulfide Saturation (S2-CSS)

option = SCSS, function = SCSS

Models calculated using PySulfSat have been benchmarked in PySulfSat:
- 'Fortin15_pss' Fortin et al. (2015) using PySulfSat by Wieser & Gleeson (2023)
- 'Liu21_pss' Liu et al. (2021) using PySulfSat by Wieser & Gleeson (2023)
- 'ONeill22_pss' O'Neill & Mavrogenes (2022) using PySulfSat by Wieser & Gleeson (2023)
- 'ONeill21_pss' O'Neill (2021) using PySulfSat by Wieser & Gleeson (2023)
- 'Smythe17_pss' Smythe et al. (2017) using PySulfSat by Wieser & Gleeson (2023)
- 'Li22_pss' Li and Zhang (2022) using PySulfSat by Wieser and Gleeson (2023)
- 'Blanchard21eq11_pss' Eq. (11) from Blanchard et al. (2021) using PySulfSat by Wieser and Gleeson (2023)
- 'Blanchard21eq12_pss' Eq. (12) from Blanchard et al. (2021) using PySulfSat by Wieser and Gleeson (2023)

The following models do not have material available in the original papers for benchmarking:
- 'ONeill21hyd' Eq. (10.34, 10.43, 10.45, 10.46, 10.49) from O'Neill (2021)
- 'ONeill21dil' Eq. (10.34, 10.43, 10.45, 10.46) including water dilution from O'Neill (2021)
- 'Liu07' Eq. (9) in Liu et al. (2007)

### 'ONeill21' Eq. (10.34, 10.43, 10.45, 10.46) excluding water dilution from O'Neill (2021)

Supplementary spreadsheet (sulfide,SCSS=ONeill21.xlsx)

Cell AR14: 7.437

Matches to 3 decimal places: Note spreadsheet uses +273 to convert to K, rather than 273.15 used in VolFe so T in spreadsheet = 1200.15 'C

In [4]:
my_models = [["SCSS", "ONeill21"],["sulfide", "ONeill21"]]
my_models = vf.make_df_and_add_model_defaults(my_models)
math.log(vf.SCSS(PT,melt_wf,models=my_models))

7.4369918428928115

### 'Thomas26_eq15_29_Wood13_eq4' Eq. (15, 29) from Thomas & Wood (2026) and eq. (4) from Wood & Wade (2013)

Thomas, R.W. and Wood, B.J., 2026. Sulfur speciation in silicate melts at high pressure. GCA 417:37-51 https://doi.org/10.1016/j.gca.2026.02.003

Wood, B.J., Wade, J. Activities and volatilities of trace components in silicate melts: a novel use of metal–silicate partitioning data. Contrib Mineral Petrol 166, 911–921 (2013). https://doi.org/10.1007/s00410-013-0896-z

Supplementary Material - Supplementary Data 2 used for benchmarking (Thomas26.xlsx) - Thomas & Wood (2026)

Cell KY14: 1693.34

Within 1%

In [5]:
my_models = [["SCSS", "Thomas26_eq15_29_Wood13_eq4"]]
my_models = vf.make_df_and_add_model_defaults(my_models)
vf.SCSS(PT,melt_wf,models=my_models)

np.float64(1701.8251983456908)

## Models for Sulfate Content at Anhydrite Saturation (S6+CAS)

option = SCAS, function = SCAS

Models calculated using PySulfSat have been benchmarked in PySulfSat:
- 'Chowdhury19_pss' Chowdhury & Dasgupta (2019) using PySulfSat by Wieser and Gleeson (2023)
- 'Zajacz19_pss' Zajacz and Tsay (2019) using PySulfSat by Wieser and Gleeson (2023)

The following models do not have material available in the original papers for benchmarking:
- 'Liu23' Eq. (4) Liu et al. (2023)

There are no other models to benchmark in VolFe.